# Review maps: Sentinel-1 seasonal composites (`s1-seasonal-rtc`)

Review the DPS output of `DPS_run_s1_composite.ipynb`: one 12-band COG per boreal tile and year, `SAR_S1_<year>_<tile>.tif`.

#### Product
+ OPERA RTC-S1 (`OPERA_L2_RTC-S1_V1`) γ⁰ backscatter, masked and reprojected per burst, merged within passes, averaged with
  equal-valid-pass weighting, Refined Lee on each seasonal mean
+ 30 m, boreal tile grid (custom Albers, 90 km tiles), nodata −9999
+ seasons within the year: **frozen** Jan 1–Mar 31 · **summer** Jun 15–Aug 31 · **shoulder** Sep 15–Oct 31
+ bands (1-based):

| season | VV mean | VH mean | VV count | VH count |
|---|---|---|---|---|
| frozen | 1 | 2 | 3 | 4 |
| summer | 5 | 6 | 7 | 8 |
| shoulder | 9 | 10 | 11 | 12 |

Means are **linear power**; everything below is shown in **dB** (10·log10). Counts are valid passes (0 = none in the tile).

#### How the multiband data are shown
1. **Summer dual-pol composite — R: VV, G: VH, B: VV/VH (dB).** The standard dual-pol display. Forest (volume scattering) has
   high VH and a low VV/VH ratio → green/yellow; open tundra, bare ground and burns have low VH and a high ratio → magenta/blue;
   open water is dark; settlements and steep slopes facing the sensor are bright.
2. **Seasonal VH composite — R: frozen, G: summer, B: shoulder (dB).** A multi-temporal display: **grey means stable**
   backscatter through the year, **colour means seasonal change**. Green = brighter in summer (leaf-on, moist), red = brighter in
   winter (e.g. frozen lakes with rough ice, wind-crusted snow), blue = brighter in autumn (wet soils, flooding).
3. **Valid-pass count (QA).** The number of passes behind each seasonal mean; few passes → noisier means.

Rescale ranges for the maps come from the 2–98th percentiles over the reviewed tiles (section 3), so the colours use the
range actually present.

Part A works anywhere the COGs can be read (static thumbnails per tile, statistics). Part B builds a MosaicJSON and an
interactive TiTiler map in the style of `icesat2_boreal/notebooks/Review_maps_v3.1_trends.ipynb`, saved to HTML.

In [ ]:
# !pip install cogeo_mosaic==8.2.0      # once per MAAP ADE workspace, as in the boreal Review notebook
import os, sys, json, importlib
import numpy as np, pandas as pd, geopandas as gpd
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt

ICESAT2_BOREAL_LIB_PATH = '/projects/my-private-bucket/code/icesat2_boreal/lib'      # maplib_folium (interactive map)
sys.path.append(ICESAT2_BOREAL_LIB_PATH)

## 1. Configuration

In [ ]:
ALGO_ID, VERSION = 's1-seasonal-rtc', 'main'
YEAR = '2020'
SUB_IDENTIFIER = 'opera_rtc_refinedlee_30m'
IDENTIFIER = f'S1_{YEAR}/{SUB_IDENTIFIER}'
USER = 'montesano'

TINDEX_DIR = f'/projects/my-public-bucket/DPS_tile_lists/S1_COMPOSITE/{VERSION}/{IDENTIFIER}'   # written by the DPS runner, section 8
TINDEX_CSV = f'{TINDEX_DIR}/S1_tindex_master.csv'
MOSAIC_JSON_LOCAL = f'{TINDEX_DIR}/S1_tindex_master_mosaic.json'
MOSAIC_JSON_S3 = MOSAIC_JSON_LOCAL.replace('/projects/my-public-bucket', f's3://maap-ops-workspace/shared/{USER}')
BOREAL_TILES_FN = '/projects/shared-buckets/montesano/databank/boreal_tiles_v004_model_ready.gpkg'
FIG_DIR = f'{TINDEX_DIR}/review'; os.makedirs(FIG_DIR, exist_ok=True)
OUT_HTML = f'/projects/my-private-bucket/code/maap_tools/maps/Review_maps_S1_composites_{YEAR}.html'

N_THUMB_TILES = 50          # tiles shown as thumbnails (all test tiles)
THUMB_PX = 500              # thumbnail size; read from COG overviews

BANDS = {'vv_frozen': 1, 'vh_frozen': 2, 'vvn_frozen': 3, 'vhn_frozen': 4,
         'vv_summer': 5, 'vh_summer': 6, 'vvn_summer': 7, 'vhn_summer': 8,
         'vv_shoulder': 9, 'vh_shoulder': 10, 'vvn_shoulder': 11, 'vhn_shoulder': 12}
SEASONS = ['frozen', 'summer', 'shoulder']

In [ ]:
tindex = pd.read_csv(TINDEX_CSV)
boreal_tiles = gpd.read_file(BOREAL_TILES_FN); boreal_tiles['tile_num'] = boreal_tiles.tile_num.astype(int)
tiles = boreal_tiles.merge(tindex, on='tile_num')
ll = tiles.to_crs(4326).geometry.centroid
tiles['lon'], tiles['lat'] = ll.x.values, ll.y.values
print(f'{len(tindex)} tiles in {TINDEX_CSV}')
# band order check against the COG band descriptions of one file
with rasterio.open(tiles.local_path.iloc[0]) as src:
    print(src.count, 'bands |', src.descriptions, '| nodata', src.nodata, '| overviews', src.overviews(1))

# Part A — Static review per tile

## 2. Read each tile at thumbnail resolution (COG overviews) and convert to dB

In [ ]:
def read_tile(path, size=THUMB_PX):
    """All 12 bands at about size x size pixels (nearest from overviews); means -> dB, counts as is; nodata -> NaN."""
    with rasterio.open(path) as src:
        scale = max(src.width, src.height) / size
        shape = (src.count, max(1, int(src.height / scale)), max(1, int(src.width / scale)))
        a = src.read(out_shape=shape, resampling=Resampling.nearest).astype('float32')
        nod = src.nodata
    a[a == nod] = np.nan
    out = {}
    for k, b in BANDS.items():
        x = a[b - 1]
        if 'n_' in k:
            out[k] = x
        else:
            with np.errstate(divide='ignore', invalid='ignore'):
                out[k] = np.where(x > 0, 10 * np.log10(x), np.nan)
    for s in SEASONS:
        out[f'ratio_{s}'] = out[f'vv_{s}'] - out[f'vh_{s}']          # VV/VH in dB
    return out

review = tiles.sort_values(['lat']).head(N_THUMB_TILES).reset_index(drop=True)
DATA = {}
for _, r in review.iterrows():
    try:
        DATA[r.tile_num] = read_tile(r.local_path)
    except Exception as e:
        print(r.tile_num, 'read failed:', e)
print(len(DATA), 'tiles read')

## 3. Per-tile statistics and the display ranges

Per tile: median dB of each mean band, share of the tile with data, median valid passes per season. Rows that stand out
(e.g. very low summer VH everywhere, missing seasons, frozen VH above summer VH over forest) are the ones to look at first.
The 2–98th percentiles over all reviewed tiles become the rescale ranges of the composites.

In [ ]:
rows = []
for t, d in DATA.items():
    r = {'tile_num': t}
    for s in SEASONS:
        for p in ('vv', 'vh'):
            r[f'{p}_{s}_med_db'] = np.nanmedian(d[f'{p}_{s}'])
        r[f'valid_frac_{s}'] = np.isfinite(d[f'vh_{s}']).mean()
        r[f'passes_{s}_med'] = np.nanmedian(np.where(d[f'vhn_{s}'] > 0, d[f'vhn_{s}'], np.nan))
    rows.append(r)
stats = review[['tile_num', 'lon', 'lat']].merge(pd.DataFrame(rows), on='tile_num').round(2)
stats.to_csv(f'{FIG_DIR}/S1_{YEAR}_tile_stats.csv', index=False)

def pct(keys, lo=2, hi=98):
    v = np.concatenate([d[k][np.isfinite(d[k])].ravel() for d in DATA.values() for k in keys])
    return [round(float(np.percentile(v, lo)), 1), round(float(np.percentile(v, hi)), 1)]
RANGES = {'vv_summer': pct(['vv_summer']), 'vh_summer': pct(['vh_summer']), 'ratio_summer': pct(['ratio_summer']),
          'vh_all': pct([f'vh_{s}' for s in SEASONS]), 'passes': [0, float(np.nanpercentile(np.concatenate(
              [d[f'vhn_{s}'][d[f'vhn_{s}'] > 0] for d in DATA.values() for s in SEASONS]), 98))]}
print('display ranges (dB; passes):', json.dumps(RANGES))
stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for s in SEASONS:
    axes[0].scatter(stats.lat, stats[f'vh_{s}_med_db'], s=14, label=s)
    axes[1].scatter(stats.lat, stats[f'passes_{s}_med'], s=14, label=s)
    axes[2].scatter(stats.lat, stats[f'valid_frac_{s}'], s=14, label=s)
for ax, t in zip(axes, ['median VH (dB)', 'median valid passes', 'share of tile with data']):
    ax.set_xlabel('tile latitude'); ax.set_title(t, loc='left', fontsize=10)
axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(f'{FIG_DIR}/S1_{YEAR}_tile_stats.png', dpi=150)

## 4. Thumbnails: the two composites and the pass counts, per tile

Columns: **summer dual-pol** (R VV, G VH, B VV/VH) · **seasonal VH** (R frozen, G summer, B shoulder) · **valid passes**
(minimum over the three seasons, so any season with few passes shows up). Tiles ordered by latitude.

In [ ]:
def stretch(x, lo, hi):
    return np.clip((x - lo) / (hi - lo), 0, 1)

def rgb_dualpol(d):
    rgb = np.dstack([stretch(d['vv_summer'], *RANGES['vv_summer']), stretch(d['vh_summer'], *RANGES['vh_summer']),
                     stretch(d['ratio_summer'], *RANGES['ratio_summer'])])
    return np.where(np.isnan(rgb), 1.0, rgb)                      # nodata white

def rgb_seasonal(d):
    lo, hi = RANGES['vh_all']
    rgb = np.dstack([stretch(d['vh_frozen'], lo, hi), stretch(d['vh_summer'], lo, hi), stretch(d['vh_shoulder'], lo, hi)])
    return np.where(np.isnan(rgb), 1.0, rgb)

def min_passes(d):
    return np.nanmin(np.dstack([d[f'vhn_{s}'] for s in SEASONS]), axis=2)

PER_FIG = 10
items = list(DATA.items())
for k in range(0, len(items), PER_FIG):
    chunk = items[k:k + PER_FIG]
    fig, axes = plt.subplots(len(chunk), 3, figsize=(10.5, 3.4 * len(chunk)), squeeze=False)
    for i, (t, d) in enumerate(chunk):
        r = review[review.tile_num == t].iloc[0]
        axes[i, 0].imshow(rgb_dualpol(d)); axes[i, 1].imshow(rgb_seasonal(d))
        im = axes[i, 2].imshow(min_passes(d), cmap='viridis', vmin=0, vmax=RANGES['passes'][1])
        axes[i, 0].set_title(f'tile {t} ({r.lat:.1f}N, {r.lon:.1f}E) · summer R VV G VH B VV/VH', fontsize=8, loc='left')
        axes[i, 1].set_title('VH R frozen G summer B shoulder', fontsize=8, loc='left')
        axes[i, 2].set_title('valid passes (min over seasons)', fontsize=8, loc='left')
        plt.colorbar(im, ax=axes[i, 2], fraction=0.046, pad=0.02)
        for a in axes[i]: a.set_xticks([]); a.set_yticks([])
    fig.tight_layout(); fig.savefig(f'{FIG_DIR}/S1_{YEAR}_thumbnails_{k // PER_FIG + 1:02d}.png', dpi=120); plt.show()

## 5. Season-to-season signal by land cover (optional check)

How VH changes between seasons for forest vs open land, from ESA WorldCover 2020 sampled on the same grid (if the LC stack
tindex is available). Forest should be brighter than open land in VH in every season and change little between summer and
shoulder; strong frozen-season drops are expected in wet open terrain.

In [ ]:
LC_TINDEX = None   # e.g. mosaiclib.LC_TINDEX_FN_DICT['c2020updated'] (tile_num, s3_path); None to skip
if LC_TINDEX:
    lc = pd.read_csv(LC_TINDEX).set_index('tile_num')
    WC = {10: 'tree', 20: 'shrub', 30: 'grass', 60: 'bare/sparse', 80: 'water', 90: 'wetland', 100: 'moss/lichen'}
    rows = []
    for t, d in DATA.items():
        if t not in lc.index: continue
        with rasterio.open(lc.loc[t, 's3_path'] if isinstance(lc.loc[t, 's3_path'], str) else lc.loc[t, 's3_path'].iloc[-1]) as src:
            c = src.read(1, out_shape=d['vh_summer'].shape, resampling=Resampling.mode)
        for code, name in WC.items():
            m = c == code
            if m.sum() < 50: continue
            rows.append({'tile_num': t, 'landcover': name, **{f'vh_{s}': np.nanmedian(d[f'vh_{s}'][m]) for s in SEASONS}})
    lcs = pd.DataFrame(rows)
    display(lcs.groupby('landcover')[[f'vh_{s}' for s in SEASONS]].median().round(1))

# Part B — Interactive map (TiTiler)

## 6. MosaicJSON from the tindex

In [ ]:
from cogeo_mosaic.mosaic import MosaicJSON
from cogeo_mosaic.backends import MosaicBackend

feats = json.loads(tiles.to_crs(4326)[['tile_num', 's3_path', 'geometry']].to_json())['features']
mosaicdata = MosaicJSON.from_features(feats, minzoom=6, maxzoom=18, accessor=lambda f: f['properties']['s3_path'])
with MosaicBackend(MOSAIC_JSON_S3, mosaic_def=mosaicdata) as mosaic:
    mosaic.write(overwrite=True)
print('wrote', MOSAIC_JSON_S3)

## 7. Tile layers

RGB layers use a TiTiler band-math expression on the linear means (`10*log10(b)` → dB) with one rescale per output band;
the single-band layers use colormaps. Ranges come from section 3.

In [ ]:
import maplib_folium
importlib.reload(maplib_folium)
from folium import TileLayer

def make_rgb_layer_dict(mosaic_json_fn, NAME, expression, rescales, SHOW_LAYER=False):
    """RGB TiTiler layer for MAP_REGISTERED_DPS_RESULTS (no colorbar). expression: 'r;g;b' in b1..b12, rescales: 3 'min,max'."""
    tiles = maplib_folium.build_tiles_with_params(None, mosaic_json_fn,
                                                  params_dict={'expression': expression, 'rescale': rescales},
                                                  titiler_endpoint='https://titiler.maap-project.org', DEBUG=False)
    layer = TileLayer(tiles=tiles, opacity=1, name=NAME, attr='MAAP', overlay=True, show=SHOW_LAYER)
    return {'layer': layer, 'cmap': 'gray', 'min_val': 0, 'max_val': 1, 'caption': NAME, 'show_cbar': False}

db = lambda b: f'10*log10(b{b})'
fmt = lambda r: f'{r[0]},{r[1]}'
S1_LAYERS = [
    make_rgb_layer_dict(MOSAIC_JSON_S3, f'S1 {YEAR} summer dual-pol (R VV, G VH, B VV/VH dB)',
                        f'{db(5)};{db(6)};{db(5)}-{db(6)}',
                        [fmt(RANGES['vv_summer']), fmt(RANGES['vh_summer']), fmt(RANGES['ratio_summer'])], SHOW_LAYER=True),
    make_rgb_layer_dict(MOSAIC_JSON_S3, f'S1 {YEAR} seasonal VH (R frozen, G summer, B shoulder dB)',
                        f'{db(2)};{db(6)};{db(10)}', [fmt(RANGES['vh_all'])] * 3),
    maplib_folium.make_tiles_layer_dict(None, MOSAIC_JSON_S3, f'S1 {YEAR} summer VH valid passes', SHOW_CBAR=True,
                                        PARAMS_DICT={'rescale': fmt(RANGES['passes']), 'bidx': '8', 'colormap_name': 'viridis'}),
    maplib_folium.make_tiles_layer_dict(None, MOSAIC_JSON_S3, f'S1 {YEAR} frozen VH valid passes', SHOW_CBAR=False,
                                        PARAMS_DICT={'rescale': fmt(RANGES['passes']), 'bidx': '4', 'colormap_name': 'viridis'}),
]

## 8. Map

In [ ]:
m = maplib_folium.MAP_REGISTERED_DPS_RESULTS(
        boreal_tiles, None, CHECK_TILES_NAME='',
        map_width=900, map_height=600, SHOW_WIDGETS=False,
        ADD_TILELAYER=S1_LAYERS)
m

## 9. Save to HTML

In [ ]:
m.save(OUT_HTML)
header = f"""
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;max-width:1400px;margin:0 auto;padding:20px 28px;">
  <h1 style="margin-bottom:6px;">Sentinel-1 seasonal composites {YEAR}</h1>
  <h3 style="margin-top:0;color:#555;font-weight:normal;">OPERA RTC-S1 &middot; 30 m boreal tiles &middot; {ALGO_ID} ({VERSION})</h3>
  <ul style="font-size:14px;line-height:1.5;color:#333;">
    <li><b>Summer dual-pol</b> &mdash; R: VV, G: VH, B: VV/VH (dB, Jun 15&ndash;Aug 31). Forest green/yellow; open land magenta/blue; water dark.</li>
    <li><b>Seasonal VH</b> &mdash; R: frozen (Jan&ndash;Mar), G: summer, B: shoulder (Sep 15&ndash;Oct 31). Grey = stable; colour = seasonal change.</li>
    <li><b>Valid passes</b> &mdash; passes behind each seasonal mean (QA).</li>
  </ul>
  <p style="font-size:12px;color:#777;">Display ranges (2&ndash;98th percentiles of the reviewed tiles): {json.dumps(RANGES)}. Built {pd.Timestamp.now():%Y-%m-%d}.</p>
</div>"""
html = open(OUT_HTML).read().replace('<body>', '<body>' + header, 1)
open(OUT_HTML, 'w').write(html)
print('saved', OUT_HTML)